In [1]:
import pandas as pd
import numpy as np
import wfdb
import ast

def load_raw_data(df, sampling_rate, path):
    if sampling_rate == 100:
        data = [wfdb.rdsamp(path+f) for f in df.filename_lr]
    else:
        data = [wfdb.rdsamp(path+f) for f in df.filename_hr]
    data = np.array([signal for signal, meta in data])
    return data

path = '/Users/shaun/Downloads/PTB_DATASET/'
sampling_rate=100

# load and convert annotation data
Y = pd.read_csv('/Users/shaun/Downloads/PTB_DATASET/ptbxl_database.csv', index_col='ecg_id')
Y.scp_codes = Y.scp_codes.apply(lambda x: ast.literal_eval(x))

# Load raw signal data
X = load_raw_data(Y, sampling_rate, path)

# Load scp_statements.csv for diagnostic aggregation
agg_df = pd.read_csv('/Users/shaun/Downloads/PTB_DATASET/scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(y_dic):
    tmp = []
    for key in y_dic.keys():
        if key in agg_df.index:
            tmp.append(agg_df.loc[key].diagnostic_class)
    return list(set(tmp))

# Apply diagnostic superclass
Y['diagnostic_superclass'] = Y.scp_codes.apply(aggregate_diagnostic)

# Split data into train and test
test_fold = 10
# Train
X_train = X[np.where(Y.strat_fold != test_fold)]
y_train = Y[(Y.strat_fold != test_fold)].diagnostic_superclass
# Test
X_test = X[np.where(Y.strat_fold == test_fold)]
y_test = Y[Y.strat_fold == test_fold].diagnostic_superclass

print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

(19601, 1000, 12) (19601,) (2198, 1000, 12) (2198,)


In [15]:
from sklearn.preprocessing import MultiLabelBinarizer

list_of_conditions = ['CD', 'STTC', 'MI', 'NORM', 'HYP']

mlb = MultiLabelBinarizer(classes=list_of_conditions)

y_train_bin = mlb.fit_transform(y_train)
y_test_bin = mlb.transform(y_test)

print(y_train_bin.shape)
# Output: (N, 5)


(19601, 5)


In [16]:
y_train_bin

array([[0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0],
       ...,
       [0, 1, 0, 0, 0],
       [0, 0, 0, 1, 0],
       [0, 0, 0, 1, 0]], shape=(19601, 5))

In [ ]:
np.unique(y_test)
list_of_conditions = ['CD', 'STTC', 'MI', 'NORM', "HYP"]

array([list([]), list(['CD']), list(['CD', 'MI', 'STTC']),
       list(['CD', 'STTC']), list(['HYP']), list(['HYP', 'CD']),
       list(['HYP', 'CD', 'MI', 'STTC']), list(['HYP', 'CD', 'STTC']),
       list(['HYP', 'MI']), list(['HYP', 'MI', 'CD']),
       list(['HYP', 'MI', 'STTC']), list(['HYP', 'NORM', 'CD']),
       list(['HYP', 'STTC']), list(['HYP', 'STTC', 'CD']),
       list(['HYP', 'STTC', 'MI', 'CD']), list(['MI']),
       list(['MI', 'CD']), list(['MI', 'STTC']), list(['NORM']),
       list(['NORM', 'CD']), list(['NORM', 'STTC']), list(['STTC']),
       list(['STTC', 'CD']), list(['STTC', 'MI', 'CD'])], dtype=object)

In [6]:
import numpy as np
import pandas as pd

values, counts = np.unique(y_test, return_counts=True)

df_freq = pd.DataFrame({
    'Category': values,
    'Count': counts
}).sort_values(by='Count', ascending=False).reset_index(drop=True)

df_freq['Percentage (%)'] = (df_freq['Count'] / len(y_test) * 100).round(2)

print(df_freq.to_string(index=False))


           Category  Count  Percentage (%)
             [NORM]    912           41.49
               [MI]    256           11.65
             [STTC]    242           11.01
               [CD]    184            8.37
           [MI, CD]    133            6.05
        [HYP, STTC]     73            3.32
         [MI, STTC]     61            2.78
              [HYP]     56            2.55
         [NORM, CD]     45            2.05
    [HYP, MI, STTC]     41            1.87
                 []     40            1.82
          [HYP, CD]     27            1.23
         [STTC, CD]     21            0.96
          [HYP, MI]     17            0.77
         [CD, STTC]     17            0.77
    [HYP, STTC, CD]     14            0.64
     [STTC, MI, CD]     13            0.59
    [HYP, CD, STTC]     11            0.50
[HYP, CD, MI, STTC]     11            0.50
     [CD, MI, STTC]      8            0.36
      [HYP, MI, CD]      5            0.23
[HYP, STTC, MI, CD]      5            0.23
       [NOR

In [13]:
empty_idx = np.where([len(x) == 0 for x in y_test])[0]
empty_idx

array([  50,   60,  211,  334,  335,  482,  488,  518,  615,  644,  677,
        846,  859,  885,  904,  947,  959,  972, 1018, 1077, 1248, 1256,
       1259, 1312, 1313, 1376, 1382, 1441, 1598, 1641, 1680, 1681, 1683,
       1712, 1769, 1791, 1837, 2097, 2126, 2167])